# EPUB Audiobook - remote chunk synthesis

This notebook synthesizes the text chunks exported by the EPUB Audiobook App
for **one patch**, using the same `VoxCPM2` model the app uses locally, and
writes the resulting `chunk_NNN.wav` files back into the same folder so the
app's **Import from Drive** button can pick them up.

It reads everything it needs from `manifest.json`, which was exported
alongside this notebook - you should not need to type any patch info by hand.

## Google Colab (recommended)
The app already uploaded this folder into **your own Google Drive** (the
account you connected). Just run the cells top to bottom: cell 2 mounts your
Drive, and the folder is a normal filesystem path from then on - no Google API
calls needed in this notebook at all.

## Kaggle
Kaggle has no native Google Drive mount. The practical flow there:
1. In the app, use **Download package locally** (instead of, or in addition
   to, exporting to Drive) to get a `.zip` of this same folder.
2. Upload that zip as a Kaggle Dataset and attach it to this notebook.
3. Skip the Drive-mount cell below, and instead set `FOLDER_PATH` to your
   Kaggle dataset input path (e.g. `/kaggle/input/<dataset-name>`).
4. After running, download the generated `chunk_NNN.wav` files from Kaggle's
   output pane and drop them into the matching Drive folder yourself (drag &
   drop in the Drive web UI, or re-upload). The app's Import step re-scans
   that Drive folder by file name either way, so it doesn't matter whether
   the files arrived via Colab or via a manual Kaggle re-upload.

In [ ]:
# Cell 1: install dependencies
!pip install -q voxcpm soundfile numpy

In [ ]:
# Cell 2: Google Colab only - mount Drive, then browse to the exported folder
# (e.g. "My Drive" > "EPUB Audiobook Exports" > "<book> - patch <n> - <timestamp>")
# and set FOLDER_PATH below. Skip this cell entirely on Kaggle.
from google.colab import drive
drive.mount('/content/drive')

FOLDER_PATH = "/content/drive/MyDrive/EPUB Audiobook Exports/<paste-the-exact-folder-name-here>"

In [ ]:
# Cell 3: Kaggle only - point at the attached dataset instead of Drive.
# FOLDER_PATH = "/kaggle/input/<dataset-name>"

In [ ]:
# Cell 4: load the manifest and the voice reference clip (if the book uses voice cloning)
import json
import os

with open(os.path.join(FOLDER_PATH, "manifest.json"), "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Patch {manifest['patch_id']} of book '{manifest['book_title']}'")
print(f"{manifest['chunk_count']} chunks, max_chars={manifest['max_chars']}")

reference_wav_path = None
prompt_text = None
if manifest.get("reference_wav"):
    reference_wav_path = os.path.join(FOLDER_PATH, manifest["reference_wav"])
    prompt_text = manifest.get("reference_transcript") or None
    print(f"Using cloned voice reference: {reference_wav_path}")

In [ ]:
# Cell 5: load the model (same model id the app uses locally)
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained(manifest.get("voxcpm_model_id", "openbmb/VoxCPM2"), load_denoiser=False)

In [ ]:
# Cell 6: synthesize every chunk that doesn't already have a matching .wav file,
# and write chunk_NNN.wav back into the same folder.
import soundfile as sf

sample_rate = model.tts_model.sample_rate

for chunk_filename in manifest["chunks"]:
    index = chunk_filename.split("_")[1].split(".")[0]  # chunk_000.txt -> 000
    out_path = os.path.join(FOLDER_PATH, f"chunk_{index}.wav")
    if os.path.exists(out_path):
        print(f"skip {chunk_filename} (already synthesized)")
        continue

    with open(os.path.join(FOLDER_PATH, chunk_filename), "r", encoding="utf-8") as f:
        text = f.read()

    kwargs = {}
    if reference_wav_path:
        kwargs["reference_wav_path"] = reference_wav_path
        if prompt_text:
            kwargs["prompt_wav_path"] = reference_wav_path
            kwargs["prompt_text"] = prompt_text

    audio = model.generate(text=text, cfg_value=2.0, inference_timesteps=10, **kwargs)
    sf.write(out_path, audio, sample_rate)
    print(f"wrote {out_path}")

print("Done. Go back to the app and click 'Import results from Drive'.")